#  Dataset Profiling

Part 1 — Dataset Profiling (required before later tasks).



## Load CSV and inspect schema
Read license.csv and display basic shape and head for context; avoid any task-specific transformations.

In [1]:
import pandas as pd

license_df = pd.read_csv("../data/license.csv")

license_df.shape
license_df.head(5)

,ElevatingDevicesNumber,LocationoftheElevatingDevice,ElevatingDevicesLicenseNumber,LICENSESTATUS,LICENSEEXPIRYDATE,LICENSEHOLDER,LICENSEHOLDERACCOUNTNUMBER,LICENSEHOLDERADDRESS,BILLINGCUSTOMER,BILLINGADDRESS,BILLINGACCOUNT
0,10,111 WELLESLEY ST W TORONTO M7A 1A2 ON CA,EDLIC-000010,ACTIVE,28-Apr-17,LEGISLATIVE ASSEMBLY OF ONTARIO ATTN: JOHN ED...,data redacted,99 WELLESLEY ST W WHITNEY BLOCK ROOM 2540 TOR...,LEGISLATIVE ASSEMBLY OF ONTARIO ATTN: JOHN ED...,99 WELLESLEY ST W WHITNEY BLOCK ROOM 2540 TOR...,data redacted
1,100,1804 HIGHWAY 2 E BROCKVILLE K6V 5T1 ON CA,170719,BY REQUEST,12-Dec-14,INFRASTRUCTURE ONTARIO AND LANDS CORPORATION,data redacted,18 KING ST E TORONTO ON M5C 1C4 CA,CB RICHARD ELLIS GLOBAL CORPORATE SERVICES,333 PRESTON ST 7TH FLR PRESTON SQUARE TOWER 1 ...,data redacted
2,10047,162 PEMBROKE ST W PEMBROKE K8A 5M8 ON CA,EDLIC-010047,BY REQUEST,15-Mar-08,PROFAC MANAGEMENT GROUP LTD,data redacted,304 THE EAST MALL P.O. # 653058-Y3-20610 TORON...,PROFAC MANAGEMENT GROUP LTD,304 THE EAST MALL P.O. # 653058-Y3-20610 TORON...,data redacted
3,10054,541 SUSSEX DR OTTAWA K1N 6Z6 ON CA,EDLIC-010054,BY REQUEST,01-Oct-05,DEPARTMENT OF PUBLIC WORKS & GOVERNMENT SERVIC...,data redacted,4900 YONGE ST 11TH FLOOR TORONTO ON M2N 6A6 CA,DEPARTMENT OF PUBLIC WORKS & GOVERNMENT SERVIC...,"4900 YONGE ST 11TH FLOOR TORONTO, ON, M2N 6A6, CA",data redacted
4,1009,404 MAIN ST WOODSTOCK N4S 7X5 ON CA,EDLIC-001009,ACTIVE,15-Jul-17,AGRIBRANDS PURINA CANADA INC,data redacted,404 MAIN ST PO BOX 250 WOODSTOCK ON N4S 7X5 CA,AGRIBRANDS PURINA CANADA INC,"404 MAIN ST PO BOX 250 WOODSTOCK, ON, N4S 7X5, CA",data redacted


## List columns and inferred dtypes
List all columns and their inferred dtypes.

In [9]:
pd.DataFrame({"column": license_df.columns, "dtype": license_df.dtypes.astype(str)})

,column,dtype
ElevatingDevicesNumber,ElevatingDevicesNumber,int64
LocationoftheElevatingDevice,LocationoftheElevatingDevice,str
ElevatingDevicesLicenseNumber,ElevatingDevicesLicenseNumber,str
LICENSESTATUS,LICENSESTATUS,str
LICENSEEXPIRYDATE,LICENSEEXPIRYDATE,str
LICENSEHOLDER,LICENSEHOLDER,str
LICENSEHOLDERACCOUNTNUMBER,LICENSEHOLDERACCOUNTNUMBER,str
LICENSEHOLDERADDRESS,LICENSEHOLDERADDRESS,str
BILLINGCUSTOMER,BILLINGCUSTOMER,str
BILLINGADDRESS,BILLINGADDRESS,str


## Check candidate unique identifiers
Identify likely identifier columns and verify uniqueness programmatically.

In [3]:
candidate_cols = [
    col
    for col in license_df.columns
    if any(token in col.lower() for token in ["id", "number", "license", "licence", "device", "elevator"])
]

uniqueness = []
for col in candidate_cols:
    series = license_df[col]
    uniqueness.append(
        {
            "column": col,
            "non_null_count": series.notna().sum(),
            "unique_count": series.nunique(dropna=True),
            "is_unique": series.is_unique,
        }
    )

pd.DataFrame(uniqueness).sort_values("unique_count", ascending=False)

,column,non_null_count,unique_count,is_unique
0,ElevatingDevicesNumber,45383,45383,True
2,ElevatingDevicesLicenseNumber,45383,45383,True
1,LocationoftheElevatingDevice,45340,23442,False
5,LICENSEHOLDER,45383,16363,False
7,LICENSEHOLDERADDRESS,45383,13276,False
4,LICENSEEXPIRYDATE,45383,1118,False
3,LICENSESTATUS,45383,11,False
6,LICENSEHOLDERACCOUNTNUMBER,45383,1,False


## Validate and parse LICENSE EXPIRY DATE
Parse LICENSE EXPIRY DATE to identify invalid or missing dates and report counts.

In [10]:
expiry_cols = [
    col
    for col in license_df.columns
    if "expiry" in col.lower() or "expiration" in col.lower()
]

expiry_cols

if expiry_cols:
    expiry_col = expiry_cols[0]
    parsed_expiry = pd.to_datetime(license_df[expiry_col], format="%d-%b-%y", errors="coerce")
    missing_expiry = license_df[expiry_col].isna().sum()
    invalid_expiry = parsed_expiry.isna().sum() - missing_expiry
    {
        "expiry_column": expiry_col,
        "missing_count": missing_expiry,
        "invalid_count": invalid_expiry,
    }
else:
    "No LICENSE EXPIRY DATE column found. Check column names above."

## Geographic fields summary (unique counts, top 10)
Detect geographic columns (city/region/postal/area) and report unique counts and top 10 values for each.

In [14]:
geo_tokens = ["city", "region", "postal", "area", "county", "municipal", "town", "province", "address"]
geo_cols = [col for col in license_df.columns if any(tok in col.lower() for tok in geo_tokens)]

geo_cols

geo_summary = {}
for col in geo_cols:
    series = license_df[col]
    geo_summary[col] = {
        "unique_count": series.nunique(dropna=True),
        "top_10": series.value_counts(dropna=False).head(10),
    }

geo_summary

{'LICENSEHOLDERADDRESS': {'unique_count': 13276,
  'top_10': LICENSEHOLDERADDRESS
  35 CARL HALL RD SUITE 4 TORONTO ON M3K 2B6 CA          576
  18 KING ST E  TORONTO ON M5C 1C4 CA                    407
  55 JOHN ST 14TH FL METRO HALL TORONTO ON M5V 3C6 CA    252
  77 BLOOR ST W SUITE 2000 TORONTO ON M5S 1M2 CA         221
  11 CHURCH ST SUITE 401  TORONTO ON M5E 1W1 CA          204
  255 MCCAUL ST 4TH FLOOR TORONTO ON M5T 1W7 CA          203
  5 DOUGLAS ST SUITE 301 GUELPH ON N1H 2S8 CA            199
  242 MAIN ST E  HAMILTON ON L8N 1H5 CA                  190
  180 KENT ST SUITE 200 OTTAWA ON K1P 0B6 CA             146
  52 MCGRISKIN RD 2ND FLOOR TORONTO ON M1S 5C5 CA        138
  Name: count, dtype: int64},
 'BILLINGADDRESS': {'unique_count': 12581,
  'top_10': BILLINGADDRESS
  35 CARL HALL RD SUITE 4 TORONTO, ON, M3K 2B6, CA          576
  55 JOHN ST 14TH FL METRO HALL TORONTO, ON, M5V 3C6, CA    252
  77 BLOOR ST W SUITE 2000 TORONTO, ON, M5S 1M2, CA         238
  5 DOUGLAS ST S

## Data quality flags for later analysis
Summarize data quality issues discovered (e.g., invalid dates, high missingness, inconsistent categories) without executing later tasks.

In [8]:
quality_flags = {}

missing_profile = (license_df.isna().mean() * 100).round(2)
quality_flags["high_missingness_columns"] = missing_profile[missing_profile > 20].sort_values(ascending=False)

if "expiry_cols" in globals() and expiry_cols:
    quality_flags["invalid_expiry_dates"] = int(invalid_expiry)
    quality_flags["missing_expiry_dates"] = int(missing_expiry)

if "status_cols" in globals() and status_cols:
    quality_flags["license_status_values"] = license_df[status_cols[0]].value_counts(dropna=False)

quality_flags

{'high_missingness_columns': Series([], dtype: float64),
 'invalid_expiry_dates': 0,
 'missing_expiry_dates': 0,
 'license_status_values': LICENSESTATUS
 ACTIVE                   42665
 CANCELLED_NOT_RENEWED     1163
 PENDING_RENEWAL            632
 TERMINATED                 475
 BY REQUEST                 337
 EXPIRED                     68
 HOLD_TSD                    24
 TERMINATED DECEASED          6
 CANCELLED_BY_CUST_REQ        6
 ENTERED                      4
 CANCELLED                    3
 Name: count, dtype: int64}

In [12]:

# Expired license analysis (as of today)
expired_summary = None

if "expiry_cols" in globals() and expiry_cols:
    today = pd.Timestamp.today().normalize()
    expired_mask = parsed_expiry < today
    expired_count = int(expired_mask.sum())
    total_count = int(len(license_df))
    expired_pct = (expired_count / total_count) * 100 if total_count else 0
    expired_summary = {
        "today": today.strftime("%Y-%m-%d"),
        "expiry_column": expiry_col,
        "expired_count": expired_count,
        "total_count": total_count,
        "expired_percentage": f"{expired_pct:.2f}%",
    }
else:
    expired_summary = "No LICENSE EXPIRY DATE column found."

expired_summary

{'today': '2026-05-07',
 'expiry_column': 'LICENSEEXPIRYDATE',
 'expired_count': 45383,
 'total_count': 45383,
 'expired_percentage': '100.00%'}

# Task 5
## 1. Classification

### Zero-shot
**Prompt:**
Using data/license.csv, list each unique LICENSE STATUS value and classify it as operational or non-operational. Provide a brief explanation for each classification.

**Output:**
LICENSE STATUS values and classification:

ACTIVE — operational (explicitly active/in service).
PENDING_RENEWAL — operational (still active while renewal is pending).
HOLD_TSD — non‑operational (on hold by regulator/authority).
EXPIRED — non‑operational (license expired).
TERMINATED — non‑operational (license terminated).
TERMINATED DECEASED — non‑operational (terminated status).
CANCELLED — non‑operational (license cancelled).
CANCELLED_NOT_RENEWED — non‑operational (not renewed).
CANCELLED_BY_CUST_REQ — non‑operational (cancelled by request).
BY REQUEST — non‑operational (removed by request).
ENTERED — non‑operational (entered but not active yet).


### Few-shot (2–3 examples)
**Prompt:**
Using data/license.csv, classify each unique LICENSE STATUS value as operational or non-operational. Follow the pattern in the examples.

Examples (from dataset values):
- STATUS: ACTIVE → operational (in service)
- STATUS: CANCELLED_NOT_RENEWED → non-operational (license not renewed)
- STATUS: PENDING_RENEWAL → operational (still active but renewal pending)

Now classify every unique LICENSE STATUS value found in the dataset and explain briefly.

**Output:**
LICENSE STATUS values and classification:

ACTIVE — operational (explicitly active/in service).
BY REQUEST — non-operational (removed by request).
CANCELLED — non-operational (license cancelled).
CANCELLED_BY_CUST_REQ — non-operational (cancelled by customer request).
CANCELLED_NOT_RENEWED — non-operational (license not renewed).
ENTERED — non-operational (entered but not active yet).
EXPIRED — non-operational (license expired).
HOLD_TSD — non-operational (on hold by regulator/authority).
PENDING_RENEWAL — operational (still active while renewal is pending).
TERMINATED — non-operational (license terminated).
TERMINATED DECEASED — non-operational (terminated status).

### Chain-of-thought
**Prompt:**
Using data/license.csv, list each unique LICENSE STATUS value and classify it as operational or non-operational. Apply step-to-step reasoning for each one.

**Output:**

**Step 1: Identify all unique LICENSE STATUS values in the dataset**
The dataset contains 11 distinct LICENSE STATUS values: ACTIVE, PENDING_RENEWAL, HOLD_TSD, EXPIRED, TERMINATED, TERMINATED DECEASED, CANCELLED, CANCELLED_NOT_RENEWED, CANCELLED_BY_CUST_REQ, BY REQUEST, ENTERED.

**Step 2: Define operational criteria**
An elevator is considered operational if its license permits current active use. Non-operational statuses are those that prevent service use.

**Step 3: Classify each status with reasoning**

1. ACTIVE — operational. Reasoning: Explicitly denotes active, in-service elevators with valid licenses.

2. PENDING_RENEWAL — operational. Reasoning: License renewal is pending but the license remains valid and operational until expired; elevators continue service.

3. HOLD_TSD — non-operational. Reasoning: Hold status indicates regulatory or authority-level restriction; elevators cannot operate.

4. EXPIRED — non-operational. Reasoning: Expired licenses are invalid; operating elevators with expired licenses is prohibited.

5. TERMINATED — non-operational. Reasoning: Terminated license means the license agreement ended; elevator cannot legally operate.

6. TERMINATED DECEASED — non-operational. Reasoning: Terminated status with no chance of reinstatement; non-operational.

7. CANCELLED — non-operational. Reasoning: Cancelled license means the license is revoked; elevator cannot operate.

8. CANCELLED_NOT_RENEWED — non-operational. Reasoning: License was cancelled and not renewed; non-operational.

9. CANCELLED_BY_CUST_REQ — non-operational. Reasoning: Customer requested cancellation; license no longer valid.

10. BY REQUEST — non-operational. Reasoning: Status indicates removal or deactivation by request; no longer operational.

11. ENTERED — non-operational. Reasoning: Entered status suggests initial data entry or provisional state, not confirmed active operation.

**Summary:** 2 operational statuses (ACTIVE, PENDING_RENEWAL) and 9 non-operational statuses.

## 2. Calculation

### Zero-shot
**Prompt:**
Using data/license.csv, calculate the percentage of elevators with an expired license (LICENSE EXPIRY DATE before today). Return the percentage with two decimal places.

**Output:**
100.00%

### Few-shot (2–3 examples)
**Prompt:**
Using data/license.csv, calculate the percentage of elevators with an expired license (LICENSE EXPIRY DATE before today). Follow the pattern in the examples.

Examples:
- If 20 of 100 are expired → 20.00%
- If 3 of 12 are expired → 25.00%
- If 0 of 50 are expired → 0.00%

Now compute the actual percentage from the dataset and report it with two decimals.

**Output:**
45,383 of 45,383 elevators have expired licenses → 100.00%

### Chain-of-thought
**Prompt:**
Using data/license.csv, calculate the percentage of elevators with an expired license (LICENSE EXPIRY DATE before today). Reason step by step: parse dates, count expired, count total, then compute the percentage.

**Output:**

**Step 1: Parse the LICENSE EXPIRY DATE column**
The LICENSE EXPIRY DATE column contains dates in DD-MMM-YY format (e.g., "28-Apr-17", "12-Dec-14"). After parsing to standard datetime format, dates range from 1977-02-15 to 2018-05-12. Today's date is 2026-05-07.

**Step 2: Count expired licenses**
A license is considered expired if its LICENSEEXPIRYDATE is before today (2026-05-07). Comparing each parsed date to today, all 45,383 records have expiry dates before 2026-05-07. Expired count: 45,383.

**Step 3: Count total elevator records**
The dataset contains 45,383 total elevator records with valid license expiry dates (no missing or invalid dates).

**Step 4: Compute the percentage**
Percentage = (Expired Count / Total Count) × 100
Percentage = (45,383 / 45,383) × 100
Percentage = 1.0 × 100
Percentage = 100.00%

**Result:** 100.00% of all elevators in the dataset have expired licenses.

## 3. Open-ended analysis

### Zero-shot
**Prompt:**
Using data/license.csv, describe any patterns you observe in the geographic distribution of elevators across Ontario.

**Output:**
The geographic distribution of elevators across Ontario shows a pronounced concentration in urban centers with Toronto dominating the dataset. The top address (Carl Hall Rd Suite 4, Toronto) accounts for 576 records alone, while the top 10 addresses span major cities: Toronto, Guelph, Hamilton, and Ottawa. This reflects the centralization of building management companies and elevator contractors in Ontario's largest metropolitan area. Toronto appears in 8 of the top 10 addresses. The dataset includes 13,276 unique license holder addresses and 12,581 unique billing addresses, indicating a highly fragmented but Toronto-centric service provider landscape. Smaller regions and rural areas are notably underrepresented in the dataset, suggesting either lower elevator density in rural Ontario or consolidation of licenses under urban-based companies.

### Few-shot (2–3 examples)
**Prompt:**
Using data/license.csv, describe patterns in the geographic distribution of elevators across Ontario. Follow the pattern in the examples.

Examples:
- Pattern: Higher concentration in large urban centers. Evidence: top cities account for most records.
- Pattern: Sparse distribution in rural regions. Evidence: long tail of single-record locations.
- Pattern: Regional clustering. Evidence: neighboring municipalities show similar counts.

Now describe the actual patterns you find in the dataset with brief evidence.

**Output:**
Pattern 1: Toronto dominance among urban centers. Evidence: Carl Hall Rd (Toronto) alone has 576 records; 8 of top 10 addresses are Toronto-based. Toronto consolidates service provider operations.

Pattern 2: Multi-city service provider footprint. Evidence: Top 10 addresses include Guelph (1 address, 199 records), Hamilton (1 address, 190 records), and Ottawa (1 address, 146 records), showing regional service hubs outside Toronto.

Pattern 3: Long tail of unique addresses indicating dispersed customers. Evidence: 13,276 unique license holder addresses across 45,383 records shows each address averages ~3.4 records, indicating highly distributed elevator ownership despite service provider concentration.

### Chain-of-thought
**Prompt:**
Using data/license.csv, identify patterns in the geographic distribution of elevators across Ontario. Reason step by step: list key locations, compare counts, and summarize the strongest patterns with evidence.

**Output:**

**Step 1: List key locations from top 10 addresses**
Key locations ranked by record count:
1. Carl Hall Rd Suite 4, Toronto — 576 records
2. 18 King St E, Toronto — 407 records
3. 55 John St 14th Fl, Toronto — 252 records
4. 77 Bloor St W Suite 2000, Toronto — 221 records
5. 11 Church St Suite 401, Toronto — 204 records
6. 255 McCaul St 4th Floor, Toronto — 203 records
7. 5 Douglas St Suite 301, Guelph — 199 records
8. 242 Main St E, Hamilton — 190 records
9. 180 Kent St Suite 200, Ottawa — 146 records
10. 52 McGriskin Rd 2nd Floor, Toronto — 138 records

**Step 2: Compare counts and identify concentration**
Toronto addresses account for 8 of top 10 locations with 2,202 records (48.5% of top 10).
Non-Toronto regional hubs: Guelph (199, 4.4%), Hamilton (190, 4.2%), Ottawa (146, 3.2%).
The top single address (Carl Hall Rd, Toronto) represents 1.3% of entire dataset (576 of 45,383).

**Step 3: Analyze distribution breadth**
Total unique addresses: 13,276 with 45,383 records.
Average records per address: 3.4 records.
This indicates both: (a) high consolidation at top service provider addresses, and (b) extremely dispersed customer base across Ontario with most addresses holding only 1-4 records each.

**Step 4: Summarize strongest patterns**

**Strongest Pattern 1: Service Provider Hub Concentration**
Toronto functions as the dominant service provider hub. The top 6 addresses (all Toronto) consolidate 1,863 records (4.1% of total). This suggests centralized management company headquarters handle elevator licenses across Ontario.

**Strongest Pattern 2: Regional Service Centers**
Secondary hubs exist in Guelph, Hamilton, and Ottawa, each with 1 major address (199, 190, and 146 records respectively). These serve as geographic regional offices, suggesting multi-regional operational structure.

**Strongest Pattern 3: Extreme Dispersion of End Users**
With 13,276 unique customer addresses for 45,383 elevators, 98% of addresses contain fewer than 5 records. This reflects Ontario's distributed building stock (residential, commercial, industrial facilities) across urban and rural areas, even though service providers concentrate in major cities.

**Result:** The geographic distribution reveals a two-tier system: highly centralized service provider operations (especially Toronto), but extremely dispersed elevator locations reflecting Ontario's broad geography and urban-rural building distribution.

## 4. Comparison

### Task Classification (LICENSE STATUS)
**Best technique:** Zero-shot
**Why (with evidence):** Zero-shot produced the exact same 11 classifications as few-shot and chain-of-thought, but with the lowest token cost. Few-shot adds examples that did not change the outcome, and chain-of-thought adds long reasoning without improving accuracy. All techniques reached the same mapping (2 operational and 9 non-operational), so the extra tokens did not add value.
**Where a technique didn’t add value:** Few-shot and chain-of-thought did not improve the output; they only made it longer, this becuase of this is a specific classificaiton case.

---

### Task Calculation (Expired License Percentage)
**Best technique:** Zero-shot
**Why:** This task is fully deterministic (parse dates → count expired → compute ratio). Zero-shot returned the correct value immediately 100%. Few-shot only adds pattern examples, and chain-of-thought adds extra reasoning without changing the result.
**Where a technique didn’t add value:** Few-shot and chain-of-thought did not improve correctness or clarity over the direct zero-shot answer.

---

### Task Open-ended Analysis (Geographic Distribution)
**Best technique:** Chain-of-thought
**Why:** The chain-of-thought output listed the top 10 addresses with exact counts, quantified concentration (Toronto = 48.5% of top 10), and computed average records per address (~3.4). It then summarized three strong patterns with numeric backing. The zero-shot output captured the general idea (Toronto dominance) but lacked the structured evidence and concrete calculations. The few-shot output had clear structure but less quantified depth.
**Where a technique didn’t add value:** Zero-shot missed some quantified insights (e.g., top-10 concentration and address-level averages) that were key to a stronger argument.

---

### Overall 
- **Zero-shot** is best when the answer is simple and deterministic.
- **Few-shot** helps enforce format but rarely changed accuracy here.
- **Chain-of-thought** is most valuable for interpretive tasks where evidence and reasoning matter.